# Making neurons fire

In the first notebook we built the **wiring diagram**: who connects to whom, and how strongly.
A wiring diagram alone does nothing. To make it *behave*, we need rules for what each neuron does with the signals it receives.

This notebook builds those rules, step by step:
1. **One neuron** receiving input, to see exactly how a neuron decides to fire.
2. **A tiny network** of four neurons, to see signals travel and get blocked.

The same rules, unchanged, later run all 166,700 neurons.

## What does a real neuron do?

A neuron is a cell with an electrical charge across its skin (the **membrane**). Inside is slightly negative compared with outside.
- When it receives excitatory input, its inside becomes **less negative**.
- If it reaches a certain level, the **threshold**, it fires a **spike**: a sharp electrical pulse lasting about a millisecond, which travels down its fibers and makes it release neurotransmitter onto its partners.
- Right after a spike, it briefly **can't fire again** (the refractory period), because the channels that create the spike need a moment to reset.
- With no input, the charge slowly drifts back to its resting level.

## The model: leaky integrate-and-fire (LIF)

Real spikes involve many kinds of ion channels. We use a simplified version that keeps only what matters for how signals spread through a network. Think of a **bucket with a small hole**:

| Bucket | Neuron |
|---|---|
| Water level | Membrane voltage |
| Pouring water in | Incoming excitatory signals (**integrate**) |
| The hole draining it | Voltage drifting back to rest (**leaky**) |
| Water reaching a line | Voltage reaching threshold → **fire** |
| Emptying the bucket | Reset after a spike |

Small inputs drain away before doing anything. Many inputs arriving **close together** can stack up past the line. That's how a neuron "decides".

## 1. The parameters

These numbers come from Shiu et al. (Nature 2024), the published fly-brain model this project follows.

- **`dt = 0.1` ms**: computers can't simulate smooth time, so we move forward in tiny steps and update everything each step.
- **`v_rest = -52` mV**: the resting voltage. The inside of a neuron is negative because ion pumps keep more positive charge outside.
- **`v_th = -45` mV**: the threshold. A neuron needs a push of 7 mV from rest to fire.
- **`v_reset = -52` mV**: where the voltage goes right after a spike.
- **`tau_m = 20` ms**: how slowly the voltage drains back to rest (the size of the hole). Bigger = slower drain = more time for inputs to add up.
- **`tau_syn = 5` ms**: how quickly one incoming signal fades away.
- **`t_ref = 2.2` ms**: the refractory period, when the neuron can't fire.
- **`w_syn = 0.275` mV**: the strength of **one** synapse. A connection with 50 synapses is 50 times stronger.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

dt = 0.1          # time step (ms)
v_rest = -52.0    # resting voltage (mV)
v_reset = -52.0   # voltage right after a spike (mV)
v_th = -45.0      # threshold: fire when voltage reaches this (mV)
tau_m = 20.0      # how slowly the bucket leaks (ms)
tau_syn = 5.0     # how quickly a synaptic tap fades (ms)
t_ref = 2.2       # refractory period after a spike (ms)
w_syn = 0.275     # strength of ONE synapse (mV)

## 2. Simulating one neuron

Our test neuron receives signals from one input neuron, which fires **randomly about 100 times per second**, through a connection with some number of synapses.

**Two variables describe the neuron at every moment:**
- `v`, the voltage: the water level.
- `g`, the synaptic input: think of it as a **tap** that opens when a signal arrives, pours strongly at first, then fades over a few milliseconds.

**The rules, as equations:**

```latex
\frac{dv}{dt} = \frac{v_{rest} - v + g}{\tau_m} \qquad \frac{dg}{dt} = -\frac{g}{\tau_{syn}}
```

In words:
- The voltage is pulled back toward rest (`v_rest - v`, the leak) and pushed up by the tap (`+ g`).
- The tap fades on its own (`-g`).

Each time step we nudge `v` and `g` a tiny bit in the direction these equations say. This is called the **Euler method**: approximate a smooth change with many small steps.

**Random input.** Real neurons fire irregularly, not like a clock. We generate input spikes randomly: in each 0.1 ms step there's a 1% chance of a spike (100 Hz × 0.1 ms). This is called **Poisson input**. The `seed` makes the randomness repeatable, so we get the same "random" spikes every run.

**After a spike:** the voltage resets and the neuron stays silent for 2.2 ms (22 steps).

In [ ]:
def simulate_one_neuron(n_synapses, input_rate_hz=100, duration_ms=300, seed=0):
    steps = int(duration_ms / dt)
    rng = np.random.default_rng(seed)
    input_spikes = rng.random(steps) < input_rate_hz * dt / 1000

    w = n_synapses * w_syn
    v, g, ref = v_rest, 0.0, 0
    voltages, spike_times = [], []

    for t in range(steps):
        if input_spikes[t]:
            g += w

        v += dt * (v_rest - v + g) / tau_m
        g -= dt * g / tau_syn

        if ref > 0:
            v = v_reset
            ref -= 1
        elif v >= v_th:
            spike_times.append(t * dt)
            v = v_reset
            ref = int(t_ref / dt)

        voltages.append(v)

    return np.array(voltages), spike_times, input_spikes

## 3. How many synapses does it take to make a neuron fire?

We run the neuron three times, with **20, 50 and 100 synapses** from the input neuron, and plot its voltage over 300 ms.
The red dashed line is the threshold; orange lines mark spikes.

**What you see:**
- **20 synapses → 0 spikes.** Each input makes a small bump that drains away. Around 120–200 ms, inputs happen to arrive close together and the bumps stack up to about -46.5 mV, almost there. This stacking is called **temporal summation**.
- **50 synapses → 7 spikes.** Stacked inputs now regularly reach threshold. The flat bits at -52 mV after each spike are the refractory period.
- **100 synapses → 19 spikes.** Strong input: one or two inputs are often enough.

**Why the voltage never goes above -45 mV:** a real spike shoots up to about +40 mV and back in a millisecond. The model skips drawing that pulse and just records *when* it happened. For how signals spread through a network, the timing of spikes is what matters, and skipping the shape makes it fast enough to simulate the whole brain.

**The key idea of the whole project:** whether a signal passes from one neuron to the next depends on **how many synapses** connect them and **how the inputs are timed**. The connectome gives the synapse counts; the model gives the rules.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 7), sharex=True)
time = np.arange(int(300 / dt)) * dt

for ax, n_syn in zip(axes, [20, 50, 100]):
    voltages, spike_times, input_spikes = simulate_one_neuron(n_syn)
    ax.plot(time, voltages, linewidth=0.8)
    ax.axhline(v_th, color="red", linestyle="--", linewidth=0.8, label="threshold")
    for st in spike_times:
        ax.axvline(st, color="orange", alpha=0.5, linewidth=0.8)
    ax.set_title(f"{n_syn} synapses → {len(spike_times)} spikes")
    ax.set_ylabel("Voltage (mV)")

axes[-1].set_xlabel("Time (ms)")
axes[0].legend()
plt.tight_layout()
plt.show()

## 4. From one neuron to a network

Two big changes:

**Neurons talk to each other.** When a neuron fires, every neuron it connects to gets input, scaled by the number of synapses (and negative if the sender is inhibitory).

**All neurons update at once.** Instead of single numbers, `v`, `g` and `ref` become **arrays**, one value per neuron. numpy updates the whole array in one operation, which is what makes simulating 166,700 neurons possible. This is called **vectorization**.

Three new ideas in the code:

- **Synaptic delay (1.8 ms).** A spike takes time to travel along the fiber and cross the synapse. So a spike sent now arrives 1.8 ms later.
- **The ring buffer.** Spikes in transit wait in a "mail sorter" with 18 slots (1.8 ms ÷ 0.1 ms), used in rotation. Each step we collect the mail in the current slot, then drop this step's new spikes into the same, now-empty slot. That slot isn't read again until 18 steps later, so every spike arrives exactly 1.8 ms after it was sent.
- **Stimulation.** To start activity, we give chosen neurons strong random input at 150 Hz. It works like **optogenetics**, a lab technique where scientists switch specific neurons on with light.

The key line is `W_mv[fired].sum(axis=0)`: take the rows of the neurons that just fired (their outgoing connections), add them up, and you get every neuron's total incoming signal. Inhibitory connections are negative, so they subtract.

In [ ]:
delay = 1.8           # synaptic delay (ms)
stim_rate = 150.0     # how fast stimulated neurons receive input (Hz)
stim_strength = 250 * w_syn   # strength of the artificial input

def simulate_network(W, stim, duration_ms=500, seed=0):
    n = W.shape[0]
    steps = int(duration_ms / dt)
    delay_steps = int(delay / dt)
    ref_steps = int(t_ref / dt)
    rng = np.random.default_rng(seed)

    v = np.full(n, v_rest)
    g = np.zeros(n)
    ref = np.zeros(n, dtype=int)
    buffer = np.zeros((delay_steps, n))
    spike_counts = np.zeros(n, dtype=int)
    W_mv = W * w_syn

    for t in range(steps):
        slot = t % delay_steps
        g += buffer[slot]
        buffer[slot] = 0

        hits = stim[rng.random(len(stim)) < stim_rate * dt / 1000]
        g[hits] += stim_strength

        v += dt * (v_rest - v + g) / tau_m
        g -= dt * g / tau_syn

        refractory = ref > 0
        v[refractory] = v_reset
        ref[refractory] -= 1

        fired = np.flatnonzero((v >= v_th) & ~refractory)
        v[fired] = v_reset
        ref[fired] = ref_steps
        spike_counts[fired] += 1

        if len(fired) > 0:
            buffer[slot] += W_mv[fired].sum(axis=0)

    return spike_counts / (duration_ms / 1000)

## 5. A tiny four-neuron network

```
A ──(100 synapses)──► B ──(100 synapses)──► C
                                            ▲
I ──(150 synapses, inhibitory)──────────────┘
```

**Experiment 1: stimulate A.** Does the signal travel down the chain to C?
**Experiment 2: stimulate A and I.** Can the inhibitory neuron I stop C?

**Results:**
| | A | B | C | I |
|---|---|---|---|---|
| Stimulate A | 178 Hz | 104 Hz | 60 Hz | 0 |
| Stimulate A and I | 142 Hz | 80 Hz | **0** | 164 Hz |

**What they mean:**
- **The signal travels, but weakens at each step** (178 → 104 → 60). Each neuron needs several inputs to stack up before it fires, so not every spike gets passed on. Real brains work the same way: signals fade unless they're strong or many neurons push together.
- **Inhibition is a veto.** With I active, C goes completely silent, even though B is still sending it signals. This is how brains make choices: one option shuts down another.
- **A fires faster than its 150 Hz input** (178 Hz). In this version, a strong input can make a neuron fire twice. We fixed this later (see the end of the notebook).
- A's rate differs between the two runs only because adding I changes how the random numbers are drawn.

**Try it yourself:** set A→B to 30 synapses (does the signal still reach C?), make I's connection -50 (can C still fire?), or add a connection from C back to A (what happens with a loop?).

In [ ]:
names = ["A", "B", "C", "I"]
A, B, C, I = 0, 1, 2, 3

W_small = np.zeros((4, 4))
W_small[A, B] = 100
W_small[B, C] = 100
W_small[I, C] = -150

rates = simulate_network(W_small, stim=np.array([A]))
print("Stimulate A:        ", dict(zip(names, rates)))

rates = simulate_network(W_small, stim=np.array([A, I]))
print("Stimulate A and I:  ", dict(zip(names, rates)))

### Cleaner printing

`np.float64(178.0)` is just how numpy 2 displays its numbers. `float(r)` turns each into a plain Python number for tidier output.

In [ ]:
print(dict(zip(names, [float(r) for r in rates])))

## 6. What we changed later, and why it matters

When we compared this simulator line by line with Shiu et al.'s original code, we found **three small differences**:

1. **Clearing input after a spike.** In Shiu's model, when a neuron fires, its synaptic input `g` is also reset to 0. Here only the voltage resets, so leftover input can make a neuron fire again right away.
2. **Freezing during the refractory period.** In Shiu's model, a refractory neuron's voltage and input don't change at all. Here they keep updating.
3. **How stimulation is delivered.** In Shiu's model, the stimulus kicks the **voltage** directly, and stimulated neurons have no refractory period, so each input event gives exactly one spike. Here it goes through `g`, which is why A fired at 178 Hz from a 150 Hz input.

These look minor in a four-neuron network, but in the whole brain they mattered: leftover input lets neurons keep re-triggering each other, which helped activity run out of control. Fixing them roughly **halved** the runaway activity in the full MaleCNS brain.

The corrected version lives in `flybrain/brain.py` (the `Brain` class) and in the web app's `web/lib/brain.ts`. This notebook keeps the original so you can see where we started.

**Summary:** a neuron model is a few simple rules: leak, integrate, fire, reset. Combined with a real wiring diagram, those rules are enough to make signals travel, fade and be blocked, just as they do in a brain.